# 06 · Training-set normalization probe

Historical diagnostic: compare a new standardized RGB model with the first 10 epochs of the completed weighted RGB baseline. The original logs are reused; the baseline is not retrained.

The returned probe reached validation image-mean water IoU **0.1803** at epoch 10, with **319 optimizer updates and one AMP-skipped step**. This supported the full standardized comparison in Notebook 07.

## 1. Baseline run and configuration

Set `BASE_RUN_NAME` to the completed Notebook 05 run. An empty value selects the latest compatible weighted run. Inherit its architecture, initialization, Adam settings, batch size, crop augmentation and weighted loss.

The probe runs for 10 epochs. Its input change is **(DN/10000 − training mean) / training standard deviation**; test and Bolivia images are not used.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import time
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from IPython.display import display, Markdown
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
BASE = ROOT / "data/sen1floods11/v1.1"
HAND = BASE / "data/flood_events/HandLabeled"

BASE_RUN_NAME = ""
baselines = sorted(p for p in (ROOT / "runs").glob("rgb_nir_weighted_*")
                   if (p / "config.json").exists() and (p / "rgb/history.csv").exists()
                   and (p / "rgb_nir/history.csv").exists())
BASE_RUN = ROOT / "runs" / BASE_RUN_NAME if BASE_RUN_NAME else baselines[-1]
base_config = json.loads((BASE_RUN / "config.json").read_text(encoding="utf-8"))
CFG = dict(base_config["training"])
CFG.update(epochs=10, input_standardization="training_valid_pixel_mean_std")
EXPERIMENTS = {"rgb_normalized": ["B4", "B3", "B2"]}
device = torch.device("cuda")
USE_AMP = CFG["amp"] and device.type == "cuda"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
RUN = ROOT / "runs" / datetime.now().strftime("rgb_normalized_probe_%Y%m%d_%H%M%S_%f")
RUN.mkdir(parents=True)
print("Baseline:", BASE_RUN)
print("Output:", RUN)
print("GPU:", torch.cuda.get_device_name(0))
splits = {}
for split in ["train", "valid"]:
    with (BASE / f"splits/flood_handlabeled/flood_{split}_data.csv").open(newline="") as f:
        splits[split] = list(csv.reader(f))
base_history = {}
for name in ["rgb", "rgb_nir"]:
    with (BASE_RUN / name / "history.csv").open(newline="", encoding="utf-8") as f:
        base_history[name] = [{k: float(v) for k, v in row.items()} for row in csv.DictReader(f)]

## 2. Baseline learning behavior

Read existing epoch logs and compare true and predicted water fractions. Training metrics were accumulated online on random crops; validation metrics use fixed full images.

In [ ]:
audit = []
for name, rows in base_history.items():
    best = max(rows, key=lambda r: r["valid_mean_water_iou"])
    audit.append(dict(experiment=name, best_epoch=int(best["epoch"]),
        last_train_global_iou=rows[-1]["train_global_water_iou"],
        best_valid_mean_iou=best["valid_mean_water_iou"],
        valid_true_water=best["valid_true_water_fraction"],
        valid_predicted_water=best["valid_predicted_water_fraction"],
        valid_precision=best["valid_precision"], minutes=sum(r["seconds"] for r in rows)/60))
def show_table(rows):
    columns = list(rows[0])
    lines = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
    for row in rows:
        lines.append("| " + " | ".join(f"{row[k]:.4f}" if isinstance(row[k], float) else str(row[k]) for k in columns) + " |")
    display(Markdown("\n".join(lines)))
show_table(audit)
fig, ax = plt.subplots(figsize=(9, 3.5), layout="constrained")
for name, rows in base_history.items():
    ax.plot([r["epoch"] for r in rows], [r["valid_predicted_water_fraction"] for r in rows], label=name + " predicted")
ax.axhline(base_history["rgb"][0]["valid_true_water_fraction"], linestyle="--", color="gray", label="True water fraction")
ax.set(xlabel="Epoch", ylabel="Fraction of valid pixels", title="Original validation water fractions")
ax.legend()
fig.savefig(RUN / "baseline_water_fractions.png", dpi=150)
plt.show()
plt.close(fig)

## 3. Training statistics and standardized inputs

Compute per-band mean and population standard deviation on training pixels with valid labels and valid B4/B3/B2/B8 observations. Accumulate in float64 and transform model inputs in float32. Reuse these fixed statistics for validation; fill missing input values with zero after standardization.

The training class counts should match the original weighted run. Percentile stretching is used only for display.

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, rows, channels, training=False, normalize=True):
        self.rows = rows
        self.channels = channels
        self.training = training
        self.epoch = 0
        self.normalize = normalize

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        image_name, label_name = self.rows[index]
        image_name = image_name.replace("_S1Hand", "_S2Hand")
        with rasterio.open(HAND / "S2Hand" / image_name) as src:
            bands = [src.descriptions.index(b) + 1 for b in ["B4", "B3", "B2", "B8"]]
            x = src.read(bands, masked=True).astype("float32").filled(np.nan)
        with rasterio.open(HAND / "LabelHand" / label_name) as src:
            y = src.read(1).astype("int64")
        y[~np.isfinite(x).all(axis=0)] = -1
        x = x / 10000.0
        if self.normalize:
            x = (x - CHANNEL_MEAN[:, None, None]) / CHANNEL_STD[:, None, None]
        x = np.nan_to_num(x)
        if self.training:
            rng = np.random.default_rng([CFG["seed"], self.epoch, index])
            size = CFG["crop_size"]
            top = int(rng.integers(0, y.shape[0] - size + 1))
            left = int(rng.integers(0, y.shape[1] - size + 1))
            x = x[:, top:top+size, left:left+size]
            y = y[top:top+size, left:left+size]
            for axis in [0, 1]:
                if rng.random() < 0.5:
                    x = np.flip(x, axis=axis + 1)
                    y = np.flip(y, axis=axis)
        return torch.from_numpy(x[:self.channels].copy()), torch.from_numpy(y.copy())


sums = np.zeros(4, dtype=np.float64)
squares = np.zeros(4, dtype=np.float64)
class_counts = np.zeros(2, dtype=np.int64)
raw_dataset = FloodDataset(splits["train"], channels=4, normalize=False)
for index in range(len(raw_dataset)):
    x_raw, label = raw_dataset[index]
    keep = label.numpy() != -1
    values = x_raw.numpy()[:, keep].astype(np.float64)
    sums += values.sum(axis=1)
    squares += np.square(values).sum(axis=1)
    class_counts += np.bincount(label.numpy()[keep], minlength=2)
count = int(class_counts.sum())
means = sums / count
stds = np.sqrt(squares / count - means**2)
CHANNEL_MEAN = means.astype(np.float32)
CHANNEL_STD = stds.astype(np.float32)
CLASS_WEIGHTS = torch.tensor(CFG["class_weights"], dtype=torch.float32, device=device)
CFG["channel_mean"] = CHANNEL_MEAN.tolist()
CFG["channel_std"] = CHANNEL_STD.tolist()
CFG["statistics_bands"] = ["B4", "B3", "B2", "B8"]
(RUN / "config.json").write_text(json.dumps({"training": CFG, "bands": EXPERIMENTS,
    "baseline": BASE_RUN.name, "torch": str(torch.__version__), "device": str(device)}, indent=2), encoding="utf-8")
print("Current / baseline class counts:", class_counts.tolist(), CFG["class_counts"])
show_table([dict(band=b, mean=float(m), std=float(s))
            for b,m,s in zip(CFG["statistics_bands"], means, stds)])

## 4. U-Net and initialization

The earlier U-Net uses encoder widths 64/128/256/512, a 1024-channel bottleneck and a symmetric decoder with transposed convolutions. Each block has two padded 3 × 3 convolutions and ReLU; a 1 × 1 head produces two logits per pixel.

Both models start from the same four-channel reference initialization. RGB uses the first three input-channel weights; all matching layers share their initial values. Each model is then trained independently.

In [ ]:
import torch
from torch import nn

class DoubleConv(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.encoders = nn.ModuleList([
            DoubleConv(in_channels, 64), DoubleConv(64, 128),
            DoubleConv(128, 256), DoubleConv(256, 512),
        ])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.upsamples = nn.ModuleList([
            nn.ConvTranspose2d(1024, 512, 2, stride=2),
            nn.ConvTranspose2d(512, 256, 2, stride=2),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.ConvTranspose2d(128, 64, 2, stride=2),
        ])
        self.decoders = nn.ModuleList([
            DoubleConv(1024, 512), DoubleConv(512, 256),
            DoubleConv(256, 128), DoubleConv(128, 64),
        ])
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for upsample, decoder, skip in zip(self.upsamples, self.decoders, reversed(skips)):
            x = upsample(x)
            x = decoder(torch.cat([skip, x], dim=1))
        return self.head(x)


torch.manual_seed(CFG["seed"])
reference = UNet(in_channels=4, num_classes=2)
initial_state = {key: value.detach().clone() for key, value in reference.state_dict().items()}
del reference

def create_model(channels):
    model = UNet(in_channels=channels, num_classes=2)
    state = dict(initial_state)
    state["encoders.0.0.weight"] = initial_state["encoders.0.0.weight"][:, :channels].clone()
    model.load_state_dict(state)
    return model.to(device)

## 5. Training and validation

Retain Notebook 05's loss normalization, full-image validation and best-mean-IoU checkpoint rule. Add counters for optimizer updates and AMP-skipped steps. Save best and last weights with their preprocessing configuration.

In [ ]:
def divide(numerator, denominator):
    return float(numerator / denominator) if denominator else float("nan")

class WaterMetrics:
    def __init__(self):
        self.matrix = np.zeros((2, 2), dtype=np.int64)
        self.image_ious = []
        self.valid_images = 0

    def update(self, prediction, target):
        for pred, truth in zip(prediction.cpu(), target.cpu()):
            valid = truth != -1
            if not valid.any():
                continue
            matrix = torch.bincount(2 * truth[valid] + pred[valid], minlength=4).reshape(2, 2).numpy()
            self.matrix += matrix
            self.valid_images += 1
            tn, fp, fn, tp = matrix.ravel()
            union = tp + fp + fn
            if union:
                self.image_ious.append(float(tp / union))

    def compute(self):
        tn, fp, fn, tp = self.matrix.ravel()
        return dict(global_water_iou=divide(tp, tp + fp + fn),
                    mean_water_iou=float(np.mean(self.image_ious)) if self.image_ious else float("nan"),
                    precision=divide(tp, tp + fp), recall=divide(tp, tp + fn),
                    f1=divide(2 * tp, 2 * tp + fp + fn),
                    true_water_fraction=divide(tp + fn, self.matrix.sum()),
                    predicted_water_fraction=divide(tp + fp, self.matrix.sum()),
                    valid_pixels=int(self.matrix.sum()), valid_images=self.valid_images,
                    images_with_water_union=len(self.image_ious))

def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    metrics = WaterMetrics()
    loss_sum, weight_sum, skipped = 0.0, 0.0, 0
    optimizer_updates, amp_skipped_steps = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        count = int((y != -1).sum())
        if count == 0:
            skipped += 1
            continue
        batch_weight = CLASS_WEIGHTS[y[y != -1]].sum()
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = F.cross_entropy(logits, y, weight=CLASS_WEIGHTS,
                                       ignore_index=-1, reduction="sum") / batch_weight
            if training:
                scaler.scale(loss).backward()
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                was_skipped = scaler.get_scale() < previous_scale
                amp_skipped_steps += int(was_skipped)
                optimizer_updates += int(not was_skipped)
        loss_sum += loss.detach().item() * batch_weight.item()
        weight_sum += batch_weight.item()
        metrics.update(logits.detach().argmax(dim=1), y)
    result = metrics.compute()
    result.update(loss=divide(loss_sum, weight_sum), skipped_batches=skipped,
                  optimizer_updates=optimizer_updates, amp_skipped_steps=amp_skipped_steps)
    return result


def write_csv(path, rows):
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def train_experiment(name, bands):
    out = RUN / name
    out.mkdir(exist_ok=True)
    dataset = FloodDataset(splits["train"], len(bands), training=True)
    train_loader = DataLoader(dataset, batch_size=CFG["batch_size"], shuffle=True,
        num_workers=CFG["num_workers"], generator=torch.Generator().manual_seed(CFG["seed"]))
    valid_loader = DataLoader(FloodDataset(splits["valid"], len(bands)), batch_size=1,
        num_workers=CFG["num_workers"])
    model = create_model(len(bands))
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best_iou, history = -1.0, []
    for epoch in range(1, CFG["epochs"] + 1):
        start = time.perf_counter()
        dataset.epoch = epoch
        train = run_epoch(model, train_loader, optimizer, scaler)
        valid = run_epoch(model, valid_loader)
        row = {"epoch": epoch, "seconds": time.perf_counter() - start,
               **{"train_" + k: v for k, v in train.items()},
               **{"valid_" + k: v for k, v in valid.items()}}
        history.append(row)
        write_csv(out / "history.csv", history)
        checkpoint = {"model": model.state_dict(), "bands": bands, "config": CFG,
                      "epoch": epoch, "validation": valid}
        torch.save(checkpoint, out / "last.pt")
        if valid["mean_water_iou"] > best_iou:
            best_iou = valid["mean_water_iou"]
            torch.save(checkpoint, out / "best.pt")
        del checkpoint
        print(f"{name} | {epoch:02d}/{CFG['epochs']} | train loss {train['loss']:.4f} | "
              f"val loss {valid['loss']:.4f} | val mean IoU {valid['mean_water_iou']:.4f} | "
              f"val predicted water {valid['predicted_water_fraction']:.2%} | "
              f"{row['seconds']:.0f}s", flush=True)
    del model, optimizer, scaler
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return history

## 6. Run the 10-epoch probe

Train a fresh RGB model with standardized inputs. The run ends at 10 epochs and is stored under `runs/rgb_normalized_probe_*`.

In [ ]:
history = train_experiment("rgb_normalized", EXPERIMENTS["rgb_normalized"])
print("Finished:", RUN)

## 7. Compare equal training budgets

Plot the original RGB log's first 10 epochs against the new standardized run. Compare losses, validation IoU and predicted water fractions under the same budget. This is an early-learning diagnostic.

In [ ]:
raw = base_history["rgb"][:CFG["epochs"]]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), layout="constrained")
for label, rows, color in [("Original RGB", raw, "#167caf"), ("Standardized RGB", history, "#df893b")]:
    epochs = [r["epoch"] for r in rows]
    axes[0].plot(epochs, [r["train_loss"] for r in rows], color=color, label=label + " train")
    axes[0].plot(epochs, [r["valid_loss"] for r in rows], color=color, linestyle="--", label=label + " valid")
    axes[1].plot(epochs, [r["valid_mean_water_iou"] for r in rows], color=color, label=label)
    axes[2].plot(epochs, [r["valid_predicted_water_fraction"] for r in rows], color=color, label=label)
axes[0].set(xlabel="Epoch", ylabel="Weighted CE", title="Training / validation loss")
axes[1].set(xlabel="Epoch", ylabel="Image-mean water IoU", ylim=(0,1), title="Validation IoU")
axes[2].axhline(raw[0]["valid_true_water_fraction"], color="gray", linestyle="--", label="True fraction")
axes[2].set(xlabel="Epoch", ylabel="Water fraction", ylim=(0,1), title="Validation prediction fraction")
for ax in axes:
    ax.legend(fontsize=7)
fig.savefig(RUN / "normalization_comparison.png", dpi=150)
plt.show()
plt.close(fig)
comparison = []
for name, rows in [("original_rgb", raw), ("normalized_rgb", history)]:
    last = rows[-1]
    comparison.append(dict(experiment=name, epoch=int(last["epoch"]),
        train_loss=last["train_loss"], valid_loss=last["valid_loss"],
        valid_mean_iou=last["valid_mean_water_iou"], valid_global_iou=last["valid_global_water_iou"],
        predicted_water=last["valid_predicted_water_fraction"]))
write_csv(RUN / "comparison.csv", comparison)
show_table(comparison)
print("New-run optimizer updates:", sum(r["train_optimizer_updates"] for r in history))
print("New-run AMP skipped steps:", sum(r["train_amp_skipped_steps"] for r in history))
print("Saved:", RUN)

The run configuration stores the full-precision channel statistics needed for inference. Keep these files together with the saved weights. The completed follow-up is [Notebook 07](07_train_standardized_rgb_vs_rgb_nir.ipynb).